In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
cd /content/drive/MyDrive/project/

/content/drive/MyDrive/project


In [ ]:
import json
import os
import zipfile
import io
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import torch

class CharCropDataset(Dataset):
    def __init__(self, path, json_file, transform=None, mode='train'):
        self.path = path

        # Cargar el JSON
        with open(json_file, "r") as f:
            self.meta = json.load(f)
        self.keys = list(self.meta.keys())

        # Transformaciones básicas para RGB (3 canales)
        if transform is None:
            if mode == 'train':
                self.transform = transforms.Compose([
                    transforms.Resize((224, 224)),  # Tamaño para EfficientNet
                    transforms.RandomRotation(5),
                    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                    transforms.ToTensor(),
                    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats
                ])
            else:
                self.transform = transforms.Compose([
                    transforms.Resize((224, 224)),  # Tamaño para EfficientNet
                    transforms.ToTensor(),
                    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats
                ])
        else:
            self.transform = transform

        # Detectar si es ZIP
        self.is_zip = path.endswith('.zip')

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        name = self.keys[idx]

        # Cargar imagen en RGB (3 canales)
        if self.is_zip:
            with zipfile.ZipFile(self.path, 'r') as zf:
                # Intentar encontrar el archivo (con o sin prefijo de ruta)
                try:
                    # Primero intenta leer directamente
                    img_bytes = zf.read(name)
                except KeyError:
                    # Si falla, busca el archivo en cualquier directorio
                    found = False
                    for zip_info in zf.infolist():
                        # Busca por nombre de archivo al final del path
                        if zip_info.filename.endswith('/' + name) or \
                          zip_info.filename == name:
                            img_bytes = zf.read(zip_info.filename)
                            found = True
                            break
                    if not found:
                        raise KeyError(f"File {name} not found in zip archive")

                img = Image.open(io.BytesIO(img_bytes)).convert('RGB')  # Cambiado a RGB
        else:
            img_path = os.path.join(self.path, name)
            img = Image.open(img_path).convert('RGB')  # Cambiado a RGB

        # Obtener bounding boxes del JSON
        bboxes_info = self.meta[name]

        crops_and_labels = []

        # Verificar si hay bounding boxes
        if not bboxes_info.get('label'):
            return crops_and_labels  # Retorna lista vacía si no hay labels

        # Obtener número de bounding boxes
        num_boxes = len(bboxes_info['label'])

        for i in range(num_boxes):
            # Extraer coordenadas para cada bbox
            left = bboxes_info["left"][i]
            top = bboxes_info["top"][i]
            width = bboxes_info["width"][i]
            height = bboxes_info["height"][i]
            label = bboxes_info["label"][i]

            # Verificar que las dimensiones sean válidas
            if width <= 0 or height <= 0:
                continue

            # Hacer crop
            right = left + width
            bottom = top + height

            # Verificar que el crop esté dentro de los límites de la imagen
            img_width, img_height = img.size
            left = max(0, left)
            top = max(0, top)
            right = min(img_width, right)
            bottom = min(img_height, bottom)

            # Si después del ajuste el bbox es inválido, saltar
            if right <= left or bottom <= top:
                continue

            crop_img = img.crop((left, top, right, bottom))

            # Aplicar transformaciones
            if self.transform:
                crop_img = self.transform(crop_img)

            # Convertir label a tensor
            label_tensor = torch.tensor(label, dtype=torch.long)

            crops_and_labels.append((crop_img, label_tensor))

        return crops_and_labels


def simple_collate_fn(batch):
    """Collate function simplificada"""
    all_images = []
    all_labels = []

    for crops_list in batch:
        for img, label in crops_list:
            all_images.append(img)
            all_labels.append(label)

    if all_images:
        # Apilar todas las imágenes
        batch_images = torch.stack(all_images, dim=0)
        batch_labels = torch.stack(all_labels, dim=0)
        return batch_images, batch_labels

    return torch.tensor([]), torch.tensor([])

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import sys
import os

# --- PATH SETUP ---
PROJECT_ROOT = os.getcwd()


# --- CONFIG ---
EPOCHS = 20           # 60 is usually enough with EfficientNet
BATCH_SIZE = 8      # Increased batch size (since we lowered image size slightly)


# NEW: Optimized Image Size (Height=64, Width=128)
# House numbers are wide. 64x128 is a very standard OCR resolution.
IMG_H, IMG_W = 224, 224

# Paths
tcdata_dir = os.path.join(PROJECT_ROOT, "tcdata")
output_dir = os.path.join(PROJECT_ROOT, "user_data", "model_data")
os.makedirs(output_dir, exist_ok=True)

# --- TRANSFORMS ---
# We use a slightly more aggressive augmentation

train_tf = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- DATASET LOADING ---
train_img = os.path.join(tcdata_dir, "mchar_train")
if not os.path.exists(train_img): train_img += ".zip"

val_img = os.path.join(tcdata_dir, "mchar_val")
if not os.path.exists(val_img): val_img += ".zip"

train_json = os.path.join(tcdata_dir, "mchar_train.json")
val_json = os.path.join(tcdata_dir, "mchar_val.json")

if not os.path.exists(train_json):
  print(f"Dataset JSON not found at {train_json}")

train_set = CharCropDataset(train_img, train_json, transform=train_tf)
val_set = CharCropDataset(val_img, val_json, transform=val_tf)

train_loader = DataLoader(train_set,
                          batch_size=BATCH_SIZE,
                          collate_fn=simple_collate_fn,
                          shuffle=True,
                          num_workers=2)

val_loader = DataLoader(val_set,
                        batch_size=BATCH_SIZE,
                        collate_fn=simple_collate_fn,
                        num_workers=2)


In [ ]:
from torch.utils.data import Subset
import numpy as np

PERCENTAGE = 0.2  # 20% of  dataset

total_samples = len(train_set)
subset_size = int(total_samples * PERCENTAGE)

indices = np.random.choice(total_samples, subset_size, replace=False)

# Crear subsets
train_subset = Subset(train_set, indices)

# Crear DataLoader con el subset
train_loader = DataLoader(train_subset,
                          batch_size=BATCH_SIZE,
                          collate_fn=simple_collate_fn,
                          shuffle=True,
                          num_workers=2)

In [ ]:

PERCENTAGE = 0.1  # 20% of  dataset

total_samples = len(train_set)
subset_size = int(total_samples * PERCENTAGE)

indices = np.random.choice(total_samples, subset_size, replace=False)

# Crear subsets
val_subset = Subset(val_set, indices)

# Crear DataLoader con el subset
val_loader = DataLoader(val_subset,
                          batch_size=BATCH_SIZE,
                          collate_fn=simple_collate_fn,
                          shuffle=True,
                          num_workers=2)

In [ ]:

import torch
import torch.nn as nn
import torchvision.models as models
import sys
import os


class OCRModel(nn.Module):
  def __init__(self, model='efficientnet_b0', num_classes=10, pretrained=True):
    super().__init__()

    # efficientnet
    self.net = models.efficientnet_b0(pretrained=True)

    # change the last layer
    num_ftrs = self.net.classifier[1].in_features

    self.net.classifier = nn.Sequential(
      nn.Dropout(p=0.2, inplace=True),
      nn.Linear(num_ftrs, num_classes)
    )

  def get_parameters_per_group(self):

      params_to_update_original = []
      params_to_update_new = []

      last_layer = 'classifier'


      for name, param in (self.net).named_parameters():
          if last_layer in name:
              params_to_update_new.append(param)
          else:
              params_to_update_original.append(param)

      return params_to_update_original, params_to_update_new

  def forward(self, x):
    return self.net(x)

In [ ]:

# --- MODEL SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = OCRModel().to(device)

params_original, params_new = model.get_parameters_per_group()

optimizer_original = torch.optim.AdamW(params_original,
                              lr = 0.0001,
                              weight_decay = 1e-4
                              )

scheduler_backbone = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_original,
    T_max=20,
    eta_min=1e-6
)

optimizer_new = torch.optim.SGD(params_new,
                          lr = 0.01,
                          momentum = 0.9,
                          weight_decay=1e-4)


# NEW: Label Smoothing
# This helps the model generalize better by not forcing 100% confidence
loss_fn = nn.CrossEntropyLoss()

Using device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 20.5M/20.5M [00:00<00:00, 94.2MB/s]


In [ ]:
batch_size=BATCH_SIZE

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_loop(dataloader, model, loss_fn, optimizers):
    model.train()
    total_samples = 0  # Contador de crops procesados

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Contar samples REALES en este batch
        batch_size_real = len(X)

        total_samples += batch_size_real

        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()

        for optimizer in optimizers:
            optimizer.step()
            optimizer.zero_grad()

        if batch % 10 == 0:
            loss_value = loss.item()
            print(f"Batch {batch}: loss: {loss_value:>7f}, samples: {total_samples}")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    total_crops = 0  # Total de crops procesados
    correct = 0
    total_loss = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            # Número REAL de crops en este batch
            batch_crops = len(X)
            total_crops += batch_crops

            pred = model(X)
            loss = loss_fn(pred, y)
            total_loss += loss.item() * batch_crops  # Ponderado por tamaño del batch

            # Calcular correctos
            correct += (pred.argmax(1) == y).sum().item()

    # Calcular métricas CORRECTAMENTE
    avg_loss = total_loss / total_crops if total_crops > 0 else 0
    accuracy = correct / total_crops if total_crops > 0 else 0

    print(f"Test Results:")
    print(f"  Total crops evaluated: {total_crops}")
    print(f"  Correct predictions: {correct}")
    print(f"  Accuracy: {100*accuracy:>0.2f}%")
    print(f"  Average loss: {avg_loss:>8f}")

    return accuracy

In [15]:
best_accuracy = 0.0
save_path = "best_model.pth"

for t in range(EPOCHS):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, model, loss_fn, [optimizer_original, optimizer_new])
    accuracy = test_loop(val_loader, model, loss_fn)
    scheduler_backbone.step()

    # save the model
    if accuracy > best_accuracy:
        best_accuracy = accuracy

        # Guardar el modelo completo
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_original_state_dict': optimizer_original.state_dict(),
            'optimizer_new_state_dict': optimizer_new.state_dict(),
            'accuracy': best_accuracy,
            'scheduler_state_dict': scheduler_backbone.state_dict(),
        }, save_path)

print("Done!")

Epoch 1
-------------------------------
Batch 0: loss: 0.378509, samples: 18
Batch 10: loss: 0.471120, samples: 183
Batch 20: loss: 0.588362, samples: 359
Batch 30: loss: 0.844684, samples: 523
Batch 40: loss: 0.557800, samples: 701
Batch 50: loss: 0.436204, samples: 876
Batch 60: loss: 0.142467, samples: 1048
Batch 70: loss: 0.490367, samples: 1222
Batch 80: loss: 0.335224, samples: 1409
Batch 90: loss: 0.137746, samples: 1596
Batch 100: loss: 0.560652, samples: 1765
Batch 110: loss: 0.283386, samples: 1935
Batch 120: loss: 0.266893, samples: 2121
Batch 130: loss: 0.576078, samples: 2296
Batch 140: loss: 0.405146, samples: 2469
Batch 150: loss: 0.706372, samples: 2651
Batch 160: loss: 0.045348, samples: 2832
Batch 170: loss: 0.407594, samples: 3009
Batch 180: loss: 0.041397, samples: 3182
Batch 190: loss: 0.508900, samples: 3349
Batch 200: loss: 0.236260, samples: 3523
Batch 210: loss: 0.107552, samples: 3695
Batch 220: loss: 0.297980, samples: 3865
Batch 230: loss: 0.645747, samples:

IndexError: Caught IndexError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 50, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataset.py", line 416, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
            ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-4250003257.py", line 45, in __getitem__
    name = self.keys[idx]
           ~~~~~~~~~^^^^^
IndexError: list index out of range
